### Interfacial Shear Layer

In [1]:
using Oceananigans
using Oceananigans.Models: ShallowWaterModel

# --------------------
# Grid & model (2-D SW, no rotation)
# --------------------
grid = RectilinearGrid(size = (128, 256),     # (Nx, Ny) — modest but smooth
                       x = (0, 2π),           # longer in x so unstable waves can fit
                       y = (-2π, 2π),
                       topology = (Periodic, Bounded, Flat))

gravitational_acceleration = 1.0
coriolis = FPlane(f = 0.0)                    # <-- no rotation

model = ShallowWaterModel(; grid,
    coriolis,
    gravitational_acceleration,
    timestepper = :RungeKutta3,
    momentum_advection = WENO(),              # stable/high-res advection
)

# --------------------
# Tanh shear initial condition (flat layer height)
# --------------------
U  = 1.0    # velocity scale
H  = 10.0   # layer thickness (constant initially)
L  = 4π/256 # shear half-width (controls tanh slope)

# Base (mean) profiles: flat height and tanh shear; v = 0 initially
h̄(x, y) = H
ū(y)    = U * tanh(y / L)
v̄(y)    = 0.0

# Small perturbation to seed the shear-layer instability
ϵ      = 1e-5            # amplitude (small!)
u′(x, y) = ϵ * randn()

# Conservative prognostic variables
uh̄(x, y) = (ū(y)) * h̄(x, y)
vh̄(x, y) = (v̄(y)) * h̄(x, y)

# "Clean" base state to compute a discrete initial ω for reference
set!(model; uh = uh̄, vh = vh̄, h = h̄)

# --------------------
# Build diagnostics and perturbation reference (from clean base)
# --------------------
uh, vh, h = model.solution
u = uh / h
v = vh / h

# Relative vorticity ζ = ∂x v − ∂y u
ω = Field(∂x(v) - ∂y(u))

# Freeze a copy of the base-state vorticity for convenience (optional)
ωⁱ = Field((Face, Face, Nothing), model.grid)
ωⁱ .= ω

# --------------------
# Now set the "true" initial condition WITH perturbation (height still flat)
# --------------------
uhⁱ(x, y) = (ū(y) + u′(x, y)) * h̄(x, y)
vhⁱ(x, y) = v̄(y) * h̄(x, y)  # still zero

set!(model; uh = uhⁱ, vh = vhⁱ, h = h̄)

# --------------------
# Simulation controls
# Pick Δt to comfortably resolve gravity waves ~ sqrt(g H)
# --------------------
Δt = 2e-3
stop_time = 50.0
simulation = Simulation(model; Δt, stop_time);

┌ Warning: The ShallowWaterModel is currently unvalidated, subject to change, and should not be used for scientific research without adequate validation.
└ @ Oceananigans.Models.ShallowWaterModels ~/.julia/packages/Oceananigans/Rb6LJ/src/Models/ShallowWaterModels/shallow_water_model.jl:129


In [2]:
# --------------------
# Output writer (NetCDF): write u, v, ω every 0.5 time units
# `write_grid=true` puts grid coordinates in the file,
# which your plotting cell reads as x_* / y_* arrays.
# --------------------
using NCDatasets

u_field = uh / h
v_field = vh / h
ω_field = Field(∂x(v_field) - ∂y(u_field))

exp_name = "tanh_shear_layer"
simulation.output_writers[:fields] = NetCDFWriter(model,
    (; u = u_field, v = v_field, ω = ω_field),
    schedule = TimeInterval(0.5),
    filename = string("../data/raw_simulation_output/", exp_name, ".nc"),
    overwrite_existing = true,
)

# --------------------
# Run
# --------------------
run!(simulation)

┌ Warning: Overwriting existing /Users/henrifdrake/code/ESS280-gfd/data/raw_simulation_output/tanh_shear_layer.nc.
└ @ OceananigansNCDatasetsExt ~/.julia/packages/Oceananigans/Rb6LJ/ext/OceananigansNCDatasetsExt.jl:1023
[ Info: Initializing simulation...
[ Info:     ... simulation initialization complete (3.141 seconds)
[ Info: Executing initial time step...
[ Info:     ... initial time step complete (2.043 seconds).
[ Info: Simulation is stopping after running for 0 seconds.
[ Info: Simulation time 50 seconds equals or exceeds stop time 50 seconds.


In [4]:
# ## Visualize the results

using Printf, CairoMakie
nothing #hide

# Read the 2-D output and build a 2×2 panel figure:
#   Row 1: total u, perturbation u
#   Row 2: total ω, perturbation ω
fig = Figure(size = (1200, 800))

axis_kwargs = (xlabel = "x", ylabel = "y")

# Row 1: u
ax_u   = Axis(fig[1, 1]; title = "Zonal velocity, u",                    axis_kwargs...)
ax_u′  = Axis(fig[1, 3]; title = "Perturbation u - u₀ (or ū)",           axis_kwargs...)

# Row 2: ω
ax_ω   = Axis(fig[2, 1]; title = "Total vorticity, ω",                   axis_kwargs...)
ax_ω′  = Axis(fig[2, 3]; title = "Perturbation ω - ω₀ (or ω̄)",          axis_kwargs...)

n = Observable(1)

ds = NCDataset(simulation.output_writers[:fields].filepath, "r")

times = ds["time"][:]

# Coordinates (swap to u/v-point coords if your grid is staggered)
x = ds["x_faa"];  y = ds["y_afa"]

# --- Total fields (reactive) ---
u_plot = @lift ds["u"][:, :, $n]
ω_plot = @lift ds["ω"][:, :, $n]

# --- Reference fields for perturbations (e.g., initial snapshot; replace with means if desired) ---
ui = ds["u"][:, :, 1]
ωi = ds["ω"][:, :, 1]

# --- Reactive perturbations ---
upert = @lift $u_plot .- ui
ωpert = @lift $ω_plot .- ωi

# --- Robust, symmetric, adaptive color ranges (ignore NaNs/Infs; avoid (0,0) collapse) ---
colorrange_sym = A -> @lift begin
    B = $A
    maxabs = mapreduce(x -> isfinite(x) ? abs(x) : 0, max, B; init = 0.0)
    maxabs = (!isfinite(maxabs) || maxabs == 0) ? 1f-9 : maxabs
    (-maxabs, maxabs)
end

cr_u′ = colorrange_sym(upert)
cr_ω′ = colorrange_sym(ωpert)

# --- Plots ---
# Total u (let Makie autorange)
hm_u  = heatmap!(ax_u,  x, y, u_plot; colorrange = (-1, 1), colormap = :balance)
Colorbar(fig[1, 2], hm_u)

# Perturbation u with adaptive symmetric range
hm_u′ = heatmap!(ax_u′, x, y, upert; colorrange = cr_u′, colormap = :balance)
Colorbar(fig[1, 4], hm_u′)

# Total ω (keep your fixed range)
hm_ω  = heatmap!(ax_ω,  x, y, ω_plot; colorrange = (-1, 1), colormap = :balance)
Colorbar(fig[2, 2], hm_ω)

# Perturbation ω with adaptive symmetric range
hm_ω′ = heatmap!(ax_ω′, x, y, ωpert; colorrange = cr_ω′, colormap = :balance)
Colorbar(fig[2, 4], hm_ω′)

# Title banner row
title = @lift @sprintf("t = %.1f", times[$n])
fig[0, 1:4] = Label(fig, title, fontsize=24, tellwidth=false)

current_figure() #hide
fig

# --- Record movie ---
frames = 1:length(times)
record(fig, string("../movies/", exp_name, ".mp4"), frames, framerate=12) do i
    n[] = i
end
nothing #hide

# Close the NetCDF when done
close(ds)

closed Dataset